# FLUKE Named Entity Recognition with OpenAI o3-2025-04-16 Reasoning Model

This notebook evaluates NER robustness using OpenAI's o3-2025-04-16 reasoning model with FLUKE linguistic modifications.

In [ ]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import ast
import json
from dotenv import load_dotenv
import numpy as np
import glob
import difflib
from scipy import stats
import time
from tqdm import tqdm

In [ ]:
load_dotenv()

In [ ]:
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## Model Configuration

Configure o3-2025-04-16 reasoning model for NER tasks.

In [ ]:
# Available o3 and o1 reasoning models
REASONING_MODELS = {
    'o3-2025-04-16': 'openai/o3-2025-04-16',
    'o1-preview': 'openai/o1-preview',
    'o1-mini': 'openai/o1-mini',
    'o1': 'openai/o1',
}

# Model selection with different reasoning strategies
REASONING_CONFIGS = {
    'standard': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'standard',
        'description': 'Standard reasoning approach with o3'
    },
    'detailed': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'detailed',
        'description': 'Detailed step-by-step reasoning with o3'
    },
    'efficient': {
        'model': 'o1-mini',
        'instruction_style': 'concise',
        'description': 'Efficient reasoning with o1-mini'
    }
}

# Select configuration
CONFIG_NAME = 'standard'  # Change to 'detailed' or 'efficient'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]
INSTRUCTION_STYLE = config['instruction_style']

print(f"Using configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Instruction style: {INSTRUCTION_STYLE}")
print(f"Description: {config['description']}")

In [ ]:
# Configure DSPy
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

## Load NER Data

In [ ]:
# Load NER dataset
ds = pd.read_json('../data/train_dev_test_data/ner/fewnerd_sample_test.json', encoding_errors='replace')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} NER samples")
print(f"Sample structure: {list(ds[0].keys())}")

In [ ]:
def remove_space(text):
    """Clean up spacing and formatting in text."""
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        # Remove multiple spaces
        cleaned = ' '.join(line.split())
        
        # Fix spacing around punctuation
        cleaned = re.sub(r'\s+([.,!?:;])', r'\1', cleaned)
        cleaned = re.sub(r'([.,!?:;])\s+', r'\1 ', cleaned)
        
        # Fix contractions
        cleaned = re.sub(r'\s*\'\s*s\b', "'s", cleaned)
        cleaned = re.sub(r'\s*n\s*\'\s*t\b', "n't", cleaned)
        cleaned = re.sub(r'\s*\'\s*ve\b', "'ve", cleaned)
        cleaned = re.sub(r'\s*\'\s*re\b', "'re", cleaned)
        cleaned = re.sub(r'\s*\'\s*ll\b', "'ll", cleaned)
        cleaned = re.sub(r'\s*\'\s*d\b', "'d", cleaned)
        cleaned = re.sub(r'\s*\'\s*m\b', "'m", cleaned)
        
        # Fix spaces around parentheses
        cleaned = re.sub(r'\(\s+', '(', cleaned)
        cleaned = re.sub(r'\s+\)', ')', cleaned)
        
        # Remove leading/trailing whitespace
        cleaned = cleaned.strip()
        
        cleaned_lines.append(cleaned)
        
    return '\n'.join(cleaned_lines)

In [ ]:
examples = [
    dspy.Example({ 
                  "text": r["text"], 
                  "label": str(r['label'])
                }).with_inputs("text")
    for r in ds
]

In [ ]:
example = examples[0]
for k, v in example.items():
    print(f"\n{k.upper()}:\n")
    print(v)

In [ ]:
def calculate_f1_ent(gold_entities, predicted_entities):
    """
    Calculates the F1 score given the true labels and predicted labels.
    """
    if predicted_entities is None:
        return 0.0, 0.0, 0.0

    true_entities = {}
    pred_entities = {}
    
    # Convert to empty list if NaN
    def handle_nan(entities):
        # If it's already a list, return it as is
        if isinstance(entities, list):
            return entities
        # Handle pandas/numpy types
        if isinstance(entities, (pd.Series, np.ndarray)):
            nan_check = pd.isna(entities)
            if isinstance(nan_check, (pd.Series, np.ndarray)):
                if nan_check.any():
                    return "[]"
            elif nan_check:
                return "[]"
        # Handle single values
        elif pd.isna(entities):
            return "[]"
        return entities

    # Handle NaN cases
    gold_entities = handle_nan(gold_entities)
    predicted_entities = handle_nan(predicted_entities)
            
    # Parse strings if needed
    if isinstance(gold_entities, str):
        gold_entities = ast.literal_eval(gold_entities)
    if isinstance(predicted_entities, str):
        predicted_entities = ast.literal_eval(predicted_entities)

    # Process gold entities
    for entity in gold_entities:
        if isinstance(entity, str):
            entity = ast.literal_eval(entity)
        if entity.get('text') is not None:
            true_entities[entity['text']] = entity['value']
        else:
            for key, value in entity.items():
                true_entities[key] = value
    
    # Process predicted entities
    for entity in predicted_entities:
        if isinstance(entity, str):
            entity = ast.literal_eval(entity)
        if entity.get('text') is not None:  
            pred_entities[entity['text']] = entity['value']
        else:
            for key, value in entity.items():
                pred_entities[key] = value

    # Calculate metrics
    true_positives = sum(1 for text in true_entities if text in pred_entities and true_entities[text] == pred_entities[text])
    false_positives = sum(1 for text in pred_entities if text not in true_entities)
    false_negatives = sum(1 for text in true_entities if text not in pred_entities)

    if true_positives == 0:
        return 0.0, 0.0, 0.0

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    f1_score = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1_score

In [ ]:
def extract_prediction(pred):
    """Extract NER predictions from o3 model output."""
    matches = re.findall(r"\[\{.*\}\]", pred)
    parsed_answer = matches[-1] if matches else ""
    if parsed_answer == "":
        return {}
    try:
        parsed_answer = ast.literal_eval(parsed_answer)
    except:
        return {}
    return parsed_answer

In [ ]:
def eval_metric(true, prediction, trace=None):
    """Evaluate NER prediction using F1 score."""
    pred = prediction.label
    
    matches = re.findall(r"\[\{.*\}\]", pred)
    parsed_answer = matches[-1] if matches else ""
    if parsed_answer == "":
        return 0.0
    try:
        parsed_answer = ast.literal_eval(parsed_answer)
    except:
        return 0.0
        
    gold_entities = ast.literal_eval(true.label)
    precision, recall, f1_score = calculate_f1_ent(gold_entities=gold_entities, predicted_entities=parsed_answer)
    return f1_score

In [ ]:
def convert_string_to_entities(entity_str):
    """Convert string representation of entities to proper format."""
    if isinstance(entity_str, str):
        try:
            # Convert string to list of dicts
            entities = ast.literal_eval(entity_str)
            # Handle nested lists by flattening
            if isinstance(entities, list):
                # Handle double nested lists
                if len(entities) > 0 and isinstance(entities[0], list):
                    entities = entities[0]
                # Handle list of dicts with text/value format
                if len(entities) > 0 and isinstance(entities[0], dict):
                    # Handle format with text/value keys
                    if 'text' in entities[0]:
                        return entities
                    # Handle format with single key-value pair
                    if len(entities[0]) == 1:
                        converted = []
                        for e in entities:
                            for text, value in e.items():
                                converted.append({'text': text, 'value': value})
                        return converted
                    # Handle format with multiple key-value pairs
                    converted = []
                    for e in entities:
                        for text, value in e.items():
                            if isinstance(value, str):
                                converted.append({'text': text, 'value': value})
                    return converted
            return entities
        except:
            return []
    return entity_str

# Evaluate the original test set

In [ ]:
from dspy.evaluate import Evaluate

## Named Entity Recognition with o3 Model

In [ ]:
class O3Ent(dspy.Signature):
    """Extract named entities from the text. Think step by step about entity boundaries, types, and context. Possible entity types: ART, BUILDING, EVENT, LOCATION, ORGANIZATION, OTHER, PERSON, PRODUCT."""
    text = dspy.InputField()
    label = dspy.OutputField(desc="The list of named entities in the text: [{'text': the text span, 'value': the entity label},].", prefix='Entities:')

In [ ]:
class O3EntModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Ent)

    def forward(self, text):
        return self.prog(text=text)

In [ ]:
o3_ent = O3EntModule()

In [ ]:
# Test with a single example
pred = o3_ent(text=example.text)
print("\nTEXT:\n")
print(example.text)
print("\nTRUE ENTITIES:\n")
print(example.label)
print("\nPREDICTION:\n")
print(pred)
print(f"\nF1 Score: {eval_metric(example, pred):.3f}")

## Evaluate Original Test Set

In [ ]:
# Use subset for testing due to o3 costs and rate limits
test_examples = examples[:100]  # Adjust size as needed

print(f"Evaluating on {len(test_examples)} examples")

evaluate = Evaluate(
    devset=test_examples, 
    metric=eval_metric, 
    num_threads=1,  # Lower for o3 models
    display_progress=True, 
    display_table=10, 
    return_outputs=True, 
    return_all_scores=True
)

results = evaluate(o3_ent)

# Save results
items = []
for sample in results[1]:
    item = {}
    sentence = sample[0]['text']
    label = sample[0]['label']
    if sample[1] == {}:
        pred = {}
    else:
        pred = sample[1]['label']
    item['text'] = sentence
    item['label'] = label
    item['pred'] = pred
    item['raw_output'] = pred  # Save full reasoning
    items.append(item)

df_result = pd.DataFrame(data=items)
df_result.to_csv(f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv', index=False)
print(f"Results saved to results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv")
print(f"Average F1 Score: {results[0]:.3f}")

## Chain-of-Thought with o3 (Enhanced Reasoning)

In [ ]:
class CoTO3Ent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(O3Ent)

    def forward(self, text):
        return self.prog(text=text)

In [ ]:
cot_o3_ent = CoTO3Ent()
pred_cot = cot_o3_ent(text=example.text)
print("\nCHAIN-OF-THOUGHT EXAMPLE:\n")
print(f"Text: {example.text}")
print("\nCOT PREDICTION:\n")
print(pred_cot)

# Evaluate by modification

In [ ]:
def evaluate_modified_set(ds, program, max_samples=50):
    """Evaluate on modified dataset with sample limit."""
    # Limit samples due to o3 cost and rate limits
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r["modified_text"]), 
                      "label": str(r['modified_label']),
                      "original_text": remove_space(r['original_text']),
                      "original_label": str(r['original_label']),
                      "index": r.get('index', 0),
                      "type": r.get('subtype', None)
                    }).with_inputs("text")
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True,
        provide_traceback=True
    )
    
    return evaluate(program)

In [ ]:
# Recreate classes for consistency
class O3Ent(dspy.Signature):
    """Extract named entities from the text. Possible entity type: ART, BUILDING, EVENT, LOCATION, ORGANIZATION, OTHER, PERSON, PRODUCT."""
    text = dspy.InputField()
    label = dspy.OutputField(desc="The list of named entities in the text: [{\"text\": the text span, \"value\": the entity label},].", prefix='Entities:')

class O3EntModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Ent)

    def forward(self, text):
        return self.prog(text=text)
        
o3_ent = O3EntModule()

In [ ]:
# Configure o3 model and load original predictions
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

# Load original predictions for comparison
original_pred_file = f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file, index_col=False)
    original_pred_ds['text'] = original_pred_ds['text'].apply(lambda x: remove_space(x.encode('utf-8').decode('unicode-escape')))
    print(f"Loaded original predictions from {original_pred_file}")
else:
    print(f"Original predictions file not found: {original_pred_file}")
    print("Please run the original evaluation first")
    original_pred_ds = None

# Get modification files (subset for testing)
json_files = glob.glob('../data/modified_data/ner/*_100.json')
test_modifications = ['typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json']
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"Testing modifications: {[f.split('/')[-1] for f in json_files]}")

for json_file in json_files:
    print(f"\nProcessing: {json_file}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    results_modified = evaluate_modified_set(data, o3_ent, max_samples=20)
    
    # Convert results to dataframe
    items = []
    for sample in results_modified[1]:
        item = {}
        if sample[1] == {}:
            pred = '[]'    
        else:
            pred = sample[1]['label']
        
        sentence = sample[0]['text']
        label = sample[0]['label'] 
        item['text'] = sentence
        item['modified_label'] = label
        pred_extracted = extract_prediction(pred)
        item['modified_pred'] = pred_extracted
        item['modified_pred'] = [{entity['text']: entity['value']} for entity in pred_extracted] if isinstance(pred_extracted, list) else []
        
        original_text = sample[0]['original_text'].encode('utf-8').decode('unicode-escape')
        item['original_text'] = original_text
        index = sample[0]['index']
        
        # Find original prediction
        if original_pred_ds is not None and index < len(original_pred_ds):
            matches = original_pred_ds['pred'].iloc[index]
            item['original_pred'] = matches if matches else '[]'
        else:
            item['original_pred'] = '[]'
            
        item['original_label'] = sample[0]['original_label']
        
        # Check if original_label is NaN and assign modified_label if it is
        if pd.isna(item['original_label']):
            item['original_label'] = item['modified_label']
        
        item['type'] = sample[0]['type']
        item['raw_output'] = pred
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results
    output_filename = f"results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Average F1: {results_modified[0]:.3f}")
    
    # Add delay to respect rate limits
    time.sleep(5)

# Aggregate results

In [ ]:
# Aggregate results across all modifications
result_files = glob.glob(f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')
aggregated_results = []

print(f"Found {len(result_files)} result files for {MODEL_NAME}-{CONFIG_NAME}")

for file in result_files:
    # Extract modification type from filename
    mod_type = file.split('-')[-1].replace('.csv', '')
    print(f"Processing: {mod_type}")
    
    try:
        # Read results file
        df = pd.read_csv(file)

        # Collect all predictions and labels for micro F1 calculation
        all_original_labels = []
        all_original_preds = []
        all_modified_labels = []
        all_modified_preds = []

        for idx, row in df.iterrows():
            # Convert string representations to proper format
            original_label = convert_string_to_entities(row['original_label'])
            original_pred = convert_string_to_entities(row['original_pred'])
            modified_label = convert_string_to_entities(row['modified_label'])
            modified_pred = convert_string_to_entities(row['modified_pred'])

            # Append to combined lists
            all_original_labels.extend(original_label)
            all_original_preds.extend(original_pred)
            all_modified_labels.extend(modified_label)
            all_modified_preds.extend(modified_pred)

        # Calculate micro F1 scores using calculate_f1_ent
        original_precision, original_recall, original_f1 = calculate_f1_ent(all_original_labels, all_original_preds)
        modified_precision, modified_recall, modified_f1 = calculate_f1_ent(all_modified_labels, all_modified_preds)
        
        # Calculate the difference between original and modified F1 scores
        difference = -round(original_f1 - modified_f1, 3)
        
        # Calculate percentage difference with respect to original F1
        pct_difference = -round((original_f1 - modified_f1) / original_f1 * 100, 2) if original_f1 != 0 else 0
        
        # Perform t-test between original and modified predictions
        try:
            t_stat, p_value = stats.ttest_ind(
                (df['original_pred'] == df['original_label']).astype(float),
                (df['modified_pred'] == df['modified_label']).astype(float)
            )
        except:
            p_value = None
        
        aggregated_results.append({
            'task': 'named_entity_recognition',
            'model': f'{MODEL_NAME}-{CONFIG_NAME}',
            'modification': mod_type,
            'original_res': round(original_f1, 3),
            'modified_res': round(modified_f1, 3),
            'difference': modified_f1 - original_f1,
            'pct_difference': pct_difference,
            'p_value': p_value,
            'original_precision': round(original_precision, 3),
            'original_recall': round(original_recall, 3),
            'modified_precision': round(modified_precision, 3),
            'modified_recall': round(modified_recall, 3),
            'samples': len(df)
        })
        
    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue

# Create final results dataframe
if aggregated_results:
    results_df = pd.DataFrame(aggregated_results)
    
    # Sort the results
    modification_order = ['temporal_bias_100', 'geographical_bias_100', 'length_bias_100', 
                         'typo_bias_100', 'capitalization_100', 'punctuation_100', 
                         'derivation_100', 'compound_word_100', 'active_to_passive_100',
                         'grammatical_role_100', 'coordinating_conjunction_100', 
                         'concept_replacement_100', 'negation_100', 'discourse_100',
                         'sentiment_100', 'casual_100', 'dialectal_100']
    
    # Only use modifications that exist in our results
    existing_mods = results_df['modification'].unique()
    modification_order = [mod for mod in modification_order if mod in existing_mods]
    
    results_df['modification'] = pd.Categorical(results_df['modification'], categories=modification_order, ordered=True)
    results_df = results_df.sort_values(by='modification')

    # Calculate averages across all modifications
    avg_original = results_df['original_res'].mean()
    avg_modified = results_df['modified_res'].mean()
    avg_difference = avg_original - avg_modified
    avg_pct_difference = results_df['pct_difference'].mean()
    avg_orig_precision = results_df['original_precision'].mean()
    avg_orig_recall = results_df['original_recall'].mean()
    avg_mod_precision = results_df['modified_precision'].mean()
    avg_mod_recall = results_df['modified_recall'].mean()

    # Add averages as a new row
    avg_row = {
        'task': 'named_entity_recognition',
        'model': f'{MODEL_NAME}-{CONFIG_NAME}',
        'modification': 'average',
        'original_res': round(avg_original, 3),
        'modified_res': round(avg_modified, 3),
        'difference': -round(avg_difference, 3),
        'pct_difference': round(avg_pct_difference, 2),
        'p_value': None,
        'original_precision': round(avg_orig_precision, 3),
        'original_recall': round(avg_orig_recall, 3),
        'modified_precision': round(avg_mod_precision, 3),
        'modified_recall': round(avg_mod_recall, 3),
        'samples': results_df['samples'].sum()
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
    print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])

    # Save aggregated results
    results_df.to_csv(f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-DP.csv', index=False)
    print(f"\nAggregated results saved to: results/ner/{MODEL_NAME}-{CONFIG_NAME}-DP.csv")

    # Apply styling to highlight performance drops
    def highlight_drops_and_significance(row):
        colors = [''] * len(row)
        if row['original_res'] > row['modified_res']:
            colors = ['background-color: red'] * len(row)
            # If p-value < 0.05, add bold text
            if 'p_value' in row and row['p_value'] is not None and row['p_value'] < 0.05:
                colors = ['background-color: red; font-weight: bold'] * len(row)
        return colors

    styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
    display(styled_df)
    
else:
    print("No results found to aggregate")

## Model Comparison and Analysis

In [ ]:
# Compare with other models if available
comparison_files = {
    'GPT-4o': 'results/ner/gpt4o-0shot-ner.csv',
    'Claude-3.5': 'results/ner/claude-0shot-ner.csv',
    'Llama-405B': 'results/ner/llama-0shot-ner.csv',
    f'{MODEL_NAME}-{CONFIG_NAME}': f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv'
}

model_f1_scores = {}
for model_name, file_path in comparison_files.items():
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            
            # Calculate F1 scores for all samples
            all_labels = []
            all_preds = []
            
            for idx, row in df.iterrows():
                labels = convert_string_to_entities(row['label'])
                preds = convert_string_to_entities(row['pred'])
                all_labels.extend(labels)
                all_preds.extend(preds)
            
            precision, recall, f1_score = calculate_f1_ent(all_labels, all_preds)
            model_f1_scores[model_name] = {
                'f1': f1_score,
                'precision': precision,
                'recall': recall
            }
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# Display comparison
if model_f1_scores:
    comparison_df = pd.DataFrame([
        {
            'Model': model,
            'F1 Score': scores['f1'],
            'Precision': scores['precision'],
            'Recall': scores['recall'],
            'Performance': f"{scores['f1']:.1%}"
        }
        for model, scores in model_f1_scores.items()
    ])
    comparison_df = comparison_df.sort_values('F1 Score', ascending=False)
    
    print("\nModel Comparison on NER:")
    print(comparison_df)

    # Highlight o3 performance
    o3_model_key = f'{MODEL_NAME}-{CONFIG_NAME}'
    if o3_model_key in model_f1_scores:
        o3_performance = model_f1_scores[o3_model_key]['f1']
        print(f"\n{o3_model_key} F1 Score: {o3_performance:.3f} ({o3_performance:.1%})")
        
        if len(model_f1_scores) > 1:
            other_models = [scores['f1'] for model, scores in model_f1_scores.items() if model != o3_model_key]
            if other_models:
                avg_others = sum(other_models) / len(other_models)
                improvement = o3_performance - avg_others
                print(f"Average of other models: {avg_others:.3f} ({avg_others:.1%})")
                print(f"Performance difference: {improvement:+.3f} ({improvement:+.1%})")

    # Style the dataframe
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]

    styled_comparison = comparison_df.style.apply(highlight_max, subset=['F1 Score'])
    display(styled_comparison)
else:
    print("No model comparison data available")

## o3 Reasoning Analysis

In [ ]:
# Analyze o3 reasoning quality (if raw outputs are available)
if 'raw_output' in df_result.columns and not df_result.empty:
    print("Sample o3 Reasoning for Named Entity Recognition:")
    print("=" * 60)
    
    for i, (idx, row) in enumerate(df_result.head(3).iterrows()):
        print(f"\nExample {i+1}:")
        print(f"Text: {row['text']}")
        print(f"True Entities: {row.get('label', row.get('modified_label', 'N/A'))}")
        
        pred_entities = extract_prediction(str(row.get('pred', row.get('modified_pred', ''))))
        print(f"Predicted Entities: {pred_entities}")
        
        print(f"Reasoning: {row['raw_output'][:800]}{'...' if len(str(row['raw_output'])) > 800 else ''}")
        print("-" * 50)

# Summary statistics
print(f"\n{MODEL_NAME}-{CONFIG_NAME} Evaluation Summary:")
print("=" * 60)

if 'results' in locals():
    print(f"Base F1 score on NER: {results[0]:.3f} ({results[0]:.1%})")

if 'aggregated_results' in locals() and aggregated_results:
    avg_robustness = sum([r['difference'] for r in aggregated_results if r['difference'] is not None]) / len([r for r in aggregated_results if r['difference'] is not None])
    print(f"Average robustness impact: {avg_robustness:+.3f}")
    print(f"Modifications tested: {len(aggregated_results)}")

print(f"\nKey insights with {MODEL_NAME}:")
print(f"- Enhanced reasoning model for entity boundary detection")
print(f"- Detailed reasoning traces show entity type analysis")
print(f"- Performance on linguistic robustness varies by modification complexity")
print(f"- Advanced reasoning helps with ambiguous entity boundaries")

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE NER Evaluation with {MODEL_NAME} Complete!")
print(f"{'='*60}")
print(f"Configuration: {config['description']}")
print(f"Files saved in results/ner/ with prefix '{MODEL_NAME}-{CONFIG_NAME}-'")
print(f"\nNext steps:")
print(f"1. Review reasoning traces for entity extraction strategies")
print(f"2. Compare robustness with other models on different modifications")
print(f"3. Analyze which linguistic changes most challenge {MODEL_NAME}")
print(f"4. Consider different reasoning configurations (detailed vs standard)")
print(f"5. Evaluate cost-benefit for NER tasks")